In [3]:
import os
import shutil
import random
import librosa
import soundfile as sf
import numpy as np
from tqdm import tqdm

# --- CONFIGURATION ---
INPUT_ROOT = "RawDataset"
OUTPUT_ROOT = "ProcessedDataset_Balanced"

# Sampling Rates & Duration
TARGET_SAMPLE_RATE = 16000
TARGET_DURATION = 5.0

# Class Limits
TRAIN_LIMIT = 1000   # Max files per class for TRAINING
TEST_LIMIT = 50      # Max files per class for TEST/VAL (Adjust this as needed)

sets_to_process = ['train', 'val', 'test']

def preprocess_audio(file_path, sr=16000, duration=5.0):
    try:
        y, _ = librosa.load(file_path, sr=sr)
        y, _ = librosa.effects.trim(y, top_db=20)
        
        # Peak Normalization
        if np.max(np.abs(y)) > 0:
            y = y / np.max(np.abs(y))
            
        # Pad or Truncate
        target_length = int(sr * duration)
        if len(y) > target_length:
            y = y[:target_length]
        else:
            padding = target_length - len(y)
            y = np.pad(y, (0, padding), mode='constant')
        return y
    except Exception:
        return None

def process_dataset():
    if os.path.exists(OUTPUT_ROOT):
        shutil.rmtree(OUTPUT_ROOT)
    os.makedirs(OUTPUT_ROOT)

    for split in sets_to_process:
        split_path = os.path.join(INPUT_ROOT, split)
        if not os.path.exists(split_path): continue

        print(f"Processing split: {split}...")
        
        for class_name in os.listdir(split_path):
            class_dir = os.path.join(split_path, class_name)
            if not os.path.isdir(class_dir): continue
            
            output_class_dir = os.path.join(OUTPUT_ROOT, split, class_name)
            os.makedirs(output_class_dir, exist_ok=True)
            
            files = [f for f in os.listdir(class_dir) if f.endswith('.wav')]
            
            # --- UPDATED BALANCING LOGIC ---
            # Set the limit based on the folder (Train vs Test/Val)
            if split == 'train':
                limit = TRAIN_LIMIT
            else:
                limit = TEST_LIMIT
            
            # Apply the limit
            if len(files) > limit:
                selected_files = random.sample(files, limit)
                # print(f"  -> {class_name}: Downsampled {len(files)} -> {limit}")
            else:
                selected_files = files
                # print(f"  -> {class_name}: Kept all {len(files)}")

            # Process files
            for filename in tqdm(selected_files, desc=f"{split}/{class_name}"):
                input_path = os.path.join(class_dir, filename)
                output_path = os.path.join(output_class_dir, filename)
                
                audio_data = preprocess_audio(input_path, sr=TARGET_SAMPLE_RATE, duration=TARGET_DURATION)
                
                if audio_data is not None:
                    sf.write(output_path, audio_data, TARGET_SAMPLE_RATE)

    print(f"\nProcessing Complete! Saved to {OUTPUT_ROOT}")

if __name__ == "__main__":
    random.seed(42)
    process_dataset()

Processing split: train...


train/Vox senilis: 100%|██████████| 1000/1000 [00:01<00:00, 502.09it/s]


Processing split: val...


val/Vox senilis: 100%|██████████| 50/50 [00:00<00:00, 470.68it/s]


Processing split: test...


test/Vox senilis: 100%|██████████| 50/50 [00:00<00:00, 516.72it/s]


Processing Complete! Saved to ProcessedDataset_Balanced


In [4]:
import os
import shutil
import random
import numpy as np
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- CONFIGURATION ---
INPUT_ROOT = "ProcessedDataset_Balanced"  # Folder from previous step
OUTPUT_ROOT = "SpectrogramDataset"

# Audio Params
SAMPLE_RATE = 16000
DURATION = 5.0 # Seconds
N_MELS = 128   # Vertical resolution of the image (freq bands)
HOP_LENGTH = 512

# Augmentation Params
# Only augment classes with fewer than this many samples
AUGMENT_TARGET = 1000 

def add_noise(y, noise_factor=0.005):
    noise = np.random.randn(len(y))
    return y + noise_factor * noise

def change_pitch(y, sr):
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=random.uniform(-2, 2))

def change_speed(y):
    return librosa.effects.time_stretch(y, rate=random.uniform(0.9, 1.1))

def save_spectrogram(y, sr, save_path):
    """
    Computes Log-Mel Spectrogram and saves it as a PNG image without axes/margins.
    """
    # 1. Compute Mel Spectrogram
    mels = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, hop_length=HOP_LENGTH)
    mels_db = librosa.power_to_db(mels, ref=np.max)

    # 2. Plot and Save (No axes, no white borders)
    plt.figure(figsize=(2.56, 2.56), dpi=100) # Results in 256x256 image
    plt.axis('off')
    plt.imshow(mels_db, aspect='auto', origin='lower', cmap='magma')
    plt.tight_layout(pad=0)
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
    plt.close()

def process_and_augment():
    if os.path.exists(OUTPUT_ROOT):
        shutil.rmtree(OUTPUT_ROOT)
    os.makedirs(OUTPUT_ROOT)

    sets = ['train', 'val', 'test']

    for split in sets:
        split_path = os.path.join(INPUT_ROOT, split)
        if not os.path.exists(split_path): continue
        
        print(f"\n--- Processing {split} set ---")

        for class_name in os.listdir(split_path):
            class_dir = os.path.join(split_path, class_name)
            if not os.path.isdir(class_dir): continue

            # Setup Output Directory
            output_class_dir = os.path.join(OUTPUT_ROOT, split, class_name)
            os.makedirs(output_class_dir, exist_ok=True)

            files = [f for f in os.listdir(class_dir) if f.endswith('.wav')]
            
            # --- AUGMENTATION LOGIC (Train Only) ---
            # We want to boost small classes to match the AUGMENT_TARGET
            augment_needed = False
            aug_factor = 1
            
            if split == 'train':
                count = len(files)
                if count < AUGMENT_TARGET:
                    augment_needed = True
                    # Calculate how many augmentations per file are needed
                    aug_factor = int(AUGMENT_TARGET / count) 
                    # print(f"  -> Augmenting {class_name}: {count} files * {aug_factor}x")

            for filename in tqdm(files, desc=f"{split}/{class_name}"):
                file_path = os.path.join(class_dir, filename)
                y, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION)
                
                # 1. Save Original Spectrogram
                base_name = os.path.splitext(filename)[0]
                save_spectrogram(y, sr, os.path.join(output_class_dir, f"{base_name}.png"))

                # 2. Generate Augmented Versions (Only for Train)
                if augment_needed:
                    for i in range(aug_factor):
                        # Randomly choose an augmentation
                        choice = random.choice(['noise', 'pitch', 'speed'])
                        
                        y_aug = y.copy()
                        if choice == 'noise':
                            y_aug = add_noise(y_aug)
                        elif choice == 'pitch':
                            y_aug = change_pitch(y_aug, sr)
                        elif choice == 'speed':
                            try:
                                y_aug = change_speed(y_aug)
                                # Fix length after time stretch
                                target_len = len(y)
                                if len(y_aug) > target_len:
                                    y_aug = y_aug[:target_len]
                                else:
                                    y_aug = np.pad(y_aug, (0, target_len - len(y_aug)))
                            except:
                                continue # Skip if stretching fails

                        save_spectrogram(y_aug, sr, os.path.join(output_class_dir, f"{base_name}_aug{i}.png"))

    print(f"\nDone! Spectrograms saved to: {os.path.abspath(OUTPUT_ROOT)}")

if __name__ == "__main__":
    # Fix random seed
    random.seed(42)
    np.random.seed(42)
    process_and_augment()


--- Processing train set ---


train/Vox senilis: 100%|██████████| 1000/1000 [02:06<00:00,  7.89it/s]



--- Processing val set ---


val/Vox senilis: 100%|██████████| 50/50 [00:05<00:00,  8.43it/s]



--- Processing test set ---


test/Vox senilis: 100%|██████████| 50/50 [00:07<00:00,  7.13it/s]


Done! Spectrograms saved to: a:\SURA\SpectrogramDataset


In [5]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

# --- CONFIGURATION ---
DATA_ROOT = "SpectrogramDataset"  # Folder with PNGs
CACHE_FILE = "cached_spectrograms.pt"  # Output file
IMAGE_SIZE = 224  # Resize to standard ResNet input size

class MemoryCachedDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None, cache_path=None):
        self.transform = transform
        self.samples = []  # This will hold (image_tensor, label_idx)
        self.classes = []
        self.class_to_idx = {}

        # 1. Check if a cache file exists to load instantly
        if cache_path and os.path.exists(cache_path):
            print(f"Loading cached dataset from {cache_path}...")
            cached_data = torch.load(cache_path)
            self.samples = cached_data['samples']
            self.classes = cached_data['classes']
            self.class_to_idx = cached_data['class_to_idx']
            print(f"Loaded {len(self.samples)} samples instantly.")
            return

        # 2. If no cache, load from folders (Slow first time)
        target_dir = os.path.join(root_dir, split)
        if not os.path.exists(target_dir):
            raise ValueError(f"Directory not found: {target_dir}")
            
        # Auto-detect classes
        self.classes = sorted([d for d in os.listdir(target_dir) if os.path.isdir(os.path.join(target_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        print(f"Found classes: {self.class_to_idx}")
        print(f"Caching '{split}' data into RAM...")

        # Loop through folders and load images
        for cls_name in self.classes:
            cls_dir = os.path.join(target_dir, cls_name)
            class_idx = self.class_to_idx[cls_name]
            
            files = [f for f in os.listdir(cls_dir) if f.endswith('.png')]
            
            for file in tqdm(files, desc=f"Loading {cls_name}"):
                img_path = os.path.join(cls_dir, file)
                
                # Load Image
                # Convert to RGB (ResNet expects 3 channels)
                img = Image.open(img_path).convert('RGB') 
                
                # Apply Transform (Resize & Convert to Tensor)
                if self.transform:
                    img_tensor = self.transform(img)
                else:
                    # Default transform if none provided
                    default_tf = transforms.Compose([
                        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                        transforms.ToTensor()
                    ])
                    img_tensor = default_tf(img)
                
                # Append to memory list
                self.samples.append((img_tensor, class_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        # Extremely fast: just list indexing
        return self.samples[idx]

    def save_cache(self, path):
        """Saves the in-memory dataset to a single .pt file"""
        print(f"Saving cache to {path}...")
        torch.save({
            'samples': self.samples,
            'classes': self.classes,
            'class_to_idx': self.class_to_idx
        }, path)
        print("Saved.")

# --- EXECUTION ---

# Define standard transforms (Normalization is usually done here too)
data_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    # Standard ImageNet normalization (good starting point for ResNet)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

if __name__ == "__main__":
    # 1. Create Train Dataset (Loads PNGs -> RAM)
    print("--- Preparing Training Set ---")
    train_ds = MemoryCachedDataset(DATA_ROOT, split='train', transform=data_transforms)
    
    # 2. Save it to disk so you don't have to wait next time
    train_ds.save_cache("train_cache.pt")

    # 3. Create Validation Set
    print("\n--- Preparing Validation Set ---")
    val_ds = MemoryCachedDataset(DATA_ROOT, split='val', transform=data_transforms)
    val_ds.save_cache("val_cache.pt")
    
    # 4. Create Test Set
    print("\n--- Preparing Test Set ---")
    test_ds = MemoryCachedDataset(DATA_ROOT, split='test', transform=data_transforms)
    test_ds.save_cache("test_cache.pt")

    print("\nDone! Use 'train_cache.pt' for training now.")

--- Preparing Training Set ---
Found classes: {'Cysts_Structural': 0, 'Dysarthia': 1, 'Laryngitis': 2, 'Vox senilis': 3, 'parkinson': 4, 'spasmodische_dysphonie': 5}
Caching 'train' data into RAM...


Loading spasmodische_dysphonie: 100%|██████████| 1224/1224 [00:22<00:00, 55.06it/s]


Saving cache to train_cache.pt...
Saved.

--- Preparing Validation Set ---
Found classes: {'Cysts_Structural': 0, 'Dysarthia': 1, 'Laryngitis': 2, 'Vox senilis': 3, 'parkinson': 4, 'spasmodische_dysphonie': 5}
Caching 'val' data into RAM...


Loading spasmodische_dysphonie: 100%|██████████| 19/19 [00:00<00:00, 55.08it/s]


Saving cache to val_cache.pt...
Saved.

--- Preparing Test Set ---
Found classes: {'Cysts_Structural': 0, 'Dysarthia': 1, 'Laryngitis': 2, 'Vox senilis': 3, 'parkinson': 4, 'spasmodische_dysphonie': 5}
Caching 'test' data into RAM...


Loading spasmodische_dysphonie: 100%|██████████| 20/20 [00:00<00:00, 55.33it/s]


Saving cache to test_cache.pt...
Saved.

Done! Use 'train_cache.pt' for training now.
